# 01 · Manchas urbanas y nodos NSRDB

**Pregunta:** ¿qué nodos NSRDB representan a cada ciudad de Tamaulipas, y qué tan
bien la representan?

> **Este notebook es autocontenido.** Corre de arriba a abajo sin necesitar los
> otros. La primera celda es el único requisito.

## Los tres notebooks

| notebook | qué responde |
|---|---|
| **01 · manchas y nodos** (este) | la GEOMETRÍA: qué es una mancha urbana, qué nodos la cubren, con qué confianza |
| **02 · clima por ciudad** | cómo se CONSTRUYE una serie climática de ciudad a partir de los nodos |
| **03 · consultas** | el MANUAL DE USO: cómo pedir los datos que quieras |

Si lo que buscas es "dame la temperatura de Victoria en junio", ve directo al
**03**.

## Fuente

Marco Geoestadístico del INEGI, edición 2025, capa `28l` (localidades
amanzanadas). La *mancha urbana* aquí es el polígono de manzanas reales, no el
municipio ni un círculo alrededor de un punto.

## Requisito

Los CSV que usa este notebook los genera:

```bash
conda run -n rs python -m urbano.construir
```

In [ ]:
# --- arranque (correr siempre primero) ---------------------------------------
# Localiza la raíz del proyecto subiendo por el árbol de directorios, para que el
# notebook funcione sin importar desde dónde se lance Jupyter.
#
# La trampa: ESTA carpeta se llama `notebooks/urbano`. Buscar un directorio
# llamado "urbano" la encuentra a ella, y Python la importa como paquete de
# espacio de nombres vacío -> "cannot import name 'config' from 'urbano'
# (unknown location)". Por eso se exige el `__init__.py`: solo el paquete real
# lo tiene.
import os, sys


def _raiz_del_proyecto(inicio="."):
    d = os.path.abspath(inicio)
    while not os.path.isfile(os.path.join(d, "urbano", "__init__.py")):
        padre = os.path.dirname(d)
        if padre == d:
            raise RuntimeError(
                f"No encuentro el paquete `urbano` subiendo desde "
                f"{os.path.abspath(inicio)}. Abre Jupyter dentro del repo.")
        d = padre
    return d


RAIZ = _raiz_del_proyecto()
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)

# Si una celda anterior ya importó el `urbano` equivocado, quedó cacheado y el
# arreglo del path no bastaría: hay que descartarlo para no exigir reiniciar el
# kernel.
for _m in [m for m in list(sys.modules) if m == "urbano" or m.startswith("urbano.")]:
    del sys.modules[_m]

import urbano
print("raíz del proyecto:", RAIZ)
print("paquete urbano   :", os.path.dirname(urbano.__file__))

# El paquete `urbano` se edita mientras el notebook está abierto, y Python NO
# recarga un módulo ya importado: la celda seguiría usando la versión vieja y
# fallaría con errores del tipo "['cve_mun'] not in index". `autoreload` vuelve
# a leer el código en cada ejecución de celda.
# (Si algo se comporta de forma rara igualmente, reinicia el kernel: autoreload
#  no rehace objetos que ya estén creados en memoria.)
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 50)

In [ ]:
from urbano import config as C
from urbano.data import manchas as M
from urbano.nodos import asignacion as A

## 1. Las manchas urbanas

63 localidades urbanas en el estado, entre ellas las 43 cabeceras municipales.

In [ ]:
urb = M.manchas_urbanas()
print(f"{len(urb)} localidades urbanas · {urb['es_cabecera'].sum()} cabeceras "
      f"municipales · {urb['area_km2'].sum():.1f} km² urbanos en total")
urb.drop(columns="geometry").head(15)

### El problema de escala

La malla NSRDB es de 0.04° → celdas de ~**18 km²**. La mayoría de las localidades
urbanas del estado son **más chicas que una sola celda**, así que un
`punto-dentro-del-polígono` puro dejaría sin ningún nodo a casi todas.

In [ ]:
CELDA_KM2 = 18.0
print((urb["area_km2"] < CELDA_KM2).sum(), "de", len(urb),
      "localidades urbanas miden menos que UNA celda NSRDB")
urb["area_km2"].describe().round(2)

## 2. Asignación nodo ↔ ciudad

Cascada de tres métodos, y cada fila declara con cuál se resolvió:

| método | condición |
|---|---|
| `dentro` | el centro del nodo cae dentro de la mancha |
| `celda` | la celda de 0.04° del nodo intersecta la mancha |
| `cercano` | la mancha no toca ninguna celda → nodo más cercano al centroide |

A cada par se le asigna un **peso** = fracción del área urbana que cae en la
celda de ese nodo. Los pesos de cada ciudad **suman 1**, así que una métrica por
ciudad es directamente `Σ pesoᵢ · métricaᵢ`.

In [ ]:
nodos = A.cargar_nodos()
pares = A.asignar(urb, nodos)
res = A.resumen(pares, urb)

print(f"{len(pares)} pares · {pares['nodo_id'].nunique()} nodos distintos "
      f"de {len(nodos)} ({pares['nodo_id'].nunique()/len(nodos):.1%} de la malla)")
print("método:", pares["metodo"].value_counts().to_dict())
print("calidad por ciudad:", res["calidad"].value_counts().to_dict())

In [ ]:
# Invariante: los pesos de cada ciudad suman exactamente 1.
pares.groupby("cvegeo")["peso"].sum().describe().round(9)

In [ ]:
res[["ciudad", "municipio", "area_km2", "n_nodos", "n_nodos_dentro",
     "peso_max", "calidad"]].head(20)

## 3. `calidad`: alta, media y baja

Un solo criterio — **cuántos nodos tienen su centro DENTRO de la mancha**:

| calidad | nodos con el centro dentro |
|---|---|
| `alta` | ≥ 3 |
| `media` | 1 – 2 |
| `baja` | 0 (la ciudad solo intersecta celdas) |

Con `baja` la "métrica de la ciudad" es en realidad la del nodo de 18 km² que la
contiene, dominado por el campo que la rodea.

In [ ]:
res.groupby("calidad")[["area_km2", "n_nodos", "n_nodos_dentro"]] \
   .agg(["count", "mean"]).round(2)

In [ ]:
# El criterio cuenta nodos, pero no mira qué fracción de la celda es ciudad.
# Por eso `media` es una categoría muy heterogénea.
res["frac_urbana_celda_dominante"] = res["area_km2"] * res["peso_max"] / CELDA_KM2
res[res["calidad"] != "baja"][
    ["ciudad", "calidad", "n_nodos_dentro", "peso_max",
     "frac_urbana_celda_dominante"]
].sort_values("frac_urbana_celda_dominante", ascending=False).round(3)

## 4. Conglomerados urbanos

Manchas separadas por menos de 2 km se agrupan. Es una alternativa data-driven a
la lista oficial de zonas metropolitanas: se recalcula sola con cada edición del
Marco Geoestadístico.

In [ ]:
conglo = (urb.groupby("nombre_conglomerado")
          .agg(localidades=("ciudad", "count"),
               area_km2=("area_km2", "sum"),
               miembros=("ciudad", lambda s: ", ".join(s)))
          .sort_values("area_km2", ascending=False))
conglo[conglo["localidades"] > 1]

### Miramar y Altamira: dos localidades, un municipio

Es la confusión más común al leer los paneles. **No son dos municipios**: son dos
*localidades urbanas* del municipio de **Altamira** (clave INEGI 003). Altamira
(`cve_loc 0001`) es la cabecera; Miramar (`0122`) no lo es, y es la más grande de
las dos.

In [ ]:
alt = urb[urb["municipio"] == "Altamira"]
print(f"{len(alt)} localidades urbanas en el municipio de Altamira")
alt[["cvegeo", "ciudad", "cve_loc", "es_cabecera", "area_km2",
     "nombre_conglomerado"]].round(2)

In [ ]:
# Comparten frontera pero NO se solapan: el área urbana total no cuenta dos
# veces la zona compartida.
print("suma de áreas  :", round(urb["area_km2"].sum(), 3), "km²")
print("área de la unión:", round(urb.geometry.union_all().area / 1e6, 3), "km²")

## 5. Mapas generales

Cuatro figuras, cada una en **dos versiones**: esquemática (fondo liso) y `_base`
(sobre mosaico de OpenStreetMap, con carreteras, calles y nombres).

```bash
conda run -n rs python -m urbano.mapas.estaticos            # ambas
conda run -n rs python -m urbano.mapas.estaticos --sin-base # solo esquemáticas
```

In [ ]:
from urbano.mapas import estaticos

# `con_base=False` evita descargar mosaicos (no necesita red).
rutas_mapas = estaticos.generar_todos(urb, nodos, pares, res, con_base=False)
for r in rutas_mapas:
    print(os.path.basename(r))

In [ ]:
from IPython.display import Image, display

for r in rutas_mapas:
    print(os.path.basename(r))
    display(Image(filename=r, width=980))

## 6. Catálogo visual: las 63 localidades, una por una

| serie | qué | orden | páginas |
|---|---|---|---|
| `cabeceras_NdeM.png` | las 43 cabeceras municipales | clave INEGI (001 → 043) | 4 |
| `otras_localidades_NdeM.png` | las 20 que no son cabecera | área descendente | 2 |

Rejilla 3 × 4. Las cabeceras se ordenan por clave —no por tamaño— porque es el
orden del catálogo oficial y permite buscar un municipio sin saber cuál es su
tamaño.

In [ ]:
from urbano.mapas import catalogo as CAT

rutas_catalogo = CAT.generar(urb, nodos, pares, con_base=False)
for r in rutas_catalogo:
    print(os.path.basename(r))

In [ ]:
for r in rutas_catalogo:
    print(os.path.basename(r))
    display(Image(filename=r, width=1000))

## 7. Mapa interactivo

Zoom hasta la manzana y consulta nodo por nodo. Necesita red para el fondo.

In [ ]:
from urbano.mapas import interactivo

ruta_html = interactivo.construir(
    urb, nodos, pares, res,
    os.path.join(C.DIR_SALIDA_MAPAS, "mapa_urbano.html"))
print(ruta_html)

## Qué sigue

- **02 · clima por ciudad** — cómo se convierten estos pesos en series climáticas.
- **03 · consultas** — cómo pedir los datos que necesitas.